# Setup

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

In [2]:
# Random state seed
RAND_STATE = 42

# Compile Data

In [3]:
DATA_DIR = Path("../dataset")
csv_files = sorted(DATA_DIR.glob('*.gz'))

channel_names = []
data_list = []
y_tensor_raw = None
onset = None
    
for i, file in enumerate(csv_files):
    # Track channel order
    channel_names.append(file.name.split('.')[0])

    df = pd.read_csv(file)

    # Confirm time col order is same as rest of file and is chronological (col 2)
    file_onset = pd.to_datetime(df.iloc[:, 2])
    if file_onset.isna().any():
        raise ValueError(f"{file.name}: {file_onset.isna().sum()} missing/invalid onset times")

    if i == 0:
        # Check the first file that it is in chronological order
        if not file_onset.is_monotonic_increasing:
            raise ValueError(f"{file.name}: rows are NOT in chronological order")
        onset = file_onset
    else:
        # Compare that it matches the first file
        if not np.array_equal(file_onset.values, onset.values):
            raise ValueError(f"{file.name}: event order differs from {csv_files[0].name}")
        
    # Extract 288 features from end and convert to numpy
    only_features = df.iloc[:,-288:]
    features_np = only_features.to_numpy().astype(np.float64)

    # Check for NaN or inf values
    if np.isnan(features_np).any():
        raise ValueError(f"{file.name}: contains NaN values")
    if np.isinf(features_np).any():
       raise ValueError(f"{file.name}: contains inf values")

    data_list.append(features_np)

    # labels will be same for every file, so just extract from the first one
    if i == 0:
        labels = df.iloc[:,0]

        # Check for missing labels
        if labels.isna().any():
            raise ValueError(f"{file.name}: {labels.isna().sum()} missing labels")

        # If 'NSEP', label is 0
        # Otherwise, label is 1 (gt10, gt30, gt60, gt100)
        y_tensor_raw = np.where(labels == 'NSEP', 0, 1)

X_tensor_raw = np.stack(data_list, axis=-1)

# Final check for bad values in stack
if not np.isfinite(X_tensor_raw).all():
    n_bad = (~np.isfinite(X_tensor_raw)).sum()
    raise ValueError(f"X_tensor_raw has {n_bad} NaN/inf values")

# Ensure that X and y lengths match
assert X_tensor_raw.shape[0] == len(y_tensor_raw), "X and y row counts differ"

# Show channel names with index
print("==================")
print("Idx: Channel Name")
print("==================")
for i, name in enumerate(channel_names):
    print(f"{i}: {name}")

Idx: Channel Name
0: F
1: Np
2: P4
3: P5
4: P6
5: Tp
6: V
7: Vx
8: Xl
9: Xs


In [4]:
class_counts = np.bincount(y_tensor_raw)
print(f"Unique labels: {', '.join(df.iloc[:,0].unique().tolist())}")
print(f"Total NSEP:    {class_counts[0]}")
print(f"Total SEP:     {class_counts[1]}")
print()
print(f"X_tensor_raw shape: {X_tensor_raw.shape}")  
print(f"y_tensor_raw shape: {y_tensor_raw.shape}")
print()
print(f"Dataset range: {onset.min().date()} to {onset.max().date()}")

Unique labels: NSEP, gt100, gt10, gt60, gt30
Total NSEP:    17625
Total SEP:     169

X_tensor_raw shape: (17794, 288, 10)
y_tensor_raw shape: (17794,)

Dataset range: 1998-01-01 to 2013-12-31


# Chronological 70/10/20 Split

In [5]:
# Create boolean masks for each class
nsep_indices = (y_tensor_raw == 0)
sep_indices = (y_tensor_raw == 1)

# Separate X and y for SEP and non-SEP
X_nsep = X_tensor_raw[nsep_indices]
y_nsep = y_tensor_raw[nsep_indices]

# Separate X and y for SEP
X_sep = X_tensor_raw[sep_indices]
y_sep = y_tensor_raw[sep_indices] 

# Split Non-SEP data chronologically into Train/Val (80%) and Test (20%)
X_nsep_tv, X_nsep_test, y_nsep_tv, y_nsep_test = train_test_split(
    X_nsep, y_nsep, test_size=0.20, random_state=RAND_STATE, shuffle=False
)

# Split the 80% block into Train (70%) and Val (10%) (of total dataset)
X_nsep_train, X_nsep_val, y_nsep_train, y_nsep_val = train_test_split(
    X_nsep_tv, y_nsep_tv, test_size=0.125, random_state=RAND_STATE, shuffle=False
)

# Split SEP data chronologically into Train/Val (80%) and Test (20%)
X_sep_tv, X_sep_test, y_sep_tv, y_sep_test = train_test_split(
    X_sep, y_sep, test_size=0.20, random_state=RAND_STATE, shuffle=False
)

# Split the 80% block into Train (70%) and Val (10%) (of total dataset)
X_sep_train, X_sep_val, y_sep_train, y_sep_val = train_test_split(
    X_sep_tv, y_sep_tv, test_size=0.125, random_state=RAND_STATE, shuffle=False
)

# Combine non-SEP and SEP partitions
X_train_raw = np.concatenate([X_nsep_train, X_sep_train], axis=0)
y_train = np.concatenate([y_nsep_train, y_sep_train], axis=0)

X_val_raw = np.concatenate([X_nsep_val,   X_sep_val],   axis=0)
y_val = np.concatenate([y_nsep_val,   y_sep_val],   axis=0)

X_test_raw = np.concatenate([X_nsep_test,  X_sep_test],  axis=0)
y_test = np.concatenate([y_nsep_test,  y_sep_test],  axis=0)

#  Shuffle each set independently (randomizes sample order within the set without mixing time partitions)
X_train_raw, y_train = shuffle(X_train_raw, y_train, random_state=RAND_STATE)
X_val_raw, y_val = shuffle(X_val_raw,   y_val,   random_state=RAND_STATE)
X_test_raw, y_test = shuffle(X_test_raw,  y_test,  random_state=RAND_STATE)


print("================")
print(f"PRE-SPLIT")
print("================")
print(f"NSEP shape: {X_nsep.shape}")
print(f"SEP shape:  {X_sep.shape}")
print(f"Ratio:      {X_nsep.shape[0] / X_sep.shape[0]:.2f}:1")
print()

for name, X, y in [("TRAINING", X_train_raw, y_train),
                   ("VALIDATION", X_val_raw, y_val),
                   ("TEST", X_test_raw, y_test)]:
    n_sep = int((y == 1).sum())
    n_nsep = int((y == 0).sum())

    print("================")
    print(f"{name} DATA")
    print("================")
    print(f"X shape:    {X.shape}")
    print(f"y shape:    {y.shape}")
    print(f"NSEP count: {n_nsep}")
    print(f"SEP count:  {n_sep}")
    print(f"Ratio:      {n_nsep / n_sep:.2f}:1")
    print()

PRE-SPLIT
NSEP shape: (17625, 288, 10)
SEP shape:  (169, 288, 10)
Ratio:      104.29:1

TRAINING DATA
X shape:    (12455, 288, 10)
y shape:    (12455,)
NSEP count: 12337
SEP count:  118
Ratio:      104.55:1

VALIDATION DATA
X shape:    (1780, 288, 10)
y shape:    (1780,)
NSEP count: 1763
SEP count:  17
Ratio:      103.71:1

TEST DATA
X shape:    (3559, 288, 10)
y shape:    (3559,)
NSEP count: 3525
SEP count:  34
Ratio:      103.68:1



### Date Split Check

In [6]:
# Load in only the onset col from CSVs
onset_df = pd.read_csv(csv_files[0], usecols=[2])
onset = pd.to_datetime(onset_df.iloc[:, 0])
idx = np.arange(len(y_tensor_raw))

def split_idx(cls_idx):
    temp, test = train_test_split(cls_idx, test_size=0.20, shuffle=False)
    train, val = train_test_split(temp, test_size=0.125, shuffle=False)
    return {"train": train, "val": val, "test": test}

splits = {0: split_idx(idx[y_tensor_raw == 0]),
          1: split_idx(idx[y_tensor_raw == 1])}


print(f"{'Split':<7} {'Class':<5} {'Samples':<8} {'Begin':<12} {'End':<12}")
print("-" * 46)
for split in ["train", "val", "test"]:
    for cls, name in [(0, "NSEP"), (1, "SEP")]:
        d = onset.iloc[splits[cls][split]]
        begin = d.min().strftime('%Y-%m-%d')
        end   = d.max().strftime('%Y-%m-%d')
        print(f"{split:<7} {name:<5} {len(d):<8} {begin:<12} {end:<12}")

Split   Class Samples  Begin        End         
----------------------------------------------
train   NSEP  12337    1998-01-01   2004-10-21  
train   SEP   118      1998-04-20   2004-11-01  
val     NSEP  1763     2004-10-21   2011-07-31  
val     SEP   17       2004-11-07   2011-03-07  
test    NSEP  3525     2011-07-31   2013-12-31  
test    SEP   34       2011-03-08   2013-12-28  


# Pre-Processing

### Hybrid Normalization

In [7]:
z_score_only = ["V", "Vx"]

X_train_norm, X_val_norm, X_test_norm = [a.astype(np.float32) for a in (X_train_raw, X_val_raw, X_test_raw)]
splits = {'train': X_train_norm, 'validation': X_val_norm, 'test': X_test_norm}

norm_params = {}
clip_report = {'validation': {}, 'test': {}}  

# Loop through every channel
for j, name in enumerate(channel_names):
    # Fit training data
    values = X_train_norm[:, :, j].astype(np.float64)
    params = {} # Track normalization parameters 

    if name not in z_score_only:
        eps = np.percentile(values[values > 0], 0.1) # Get the smallest offset

        # Shift up if negative or zeros 
        min_value = values.min()
        shifted = bool(min_value + eps <= 0)
        if shifted:
            centred = values - min_value 
            delta = np.percentile(centred[centred > 0], 0.1)      
            eps = abs(min_value) + delta

        # Determine offset
        floor = (values + eps).min()

        # Apply log
        values = np.log10(np.maximum(values + eps, floor))

        params.update(method='log_zscore', eps=eps, floor=floor, shifted=shifted)

    elif name in z_score_only:
        params.update(method='zscore')

    else:
        raise ValueError(f"No normalization method defined for channel {j} ({name})")

    # Get parameters and save to dict
    params['mean'], params['std'] = values.mean(), values.std()    # train-only stats
    norm_params[name] = params

    # Apply normalization to all sets
    for split_name, X in splits.items():
        x = X[:, :, j].astype(np.float64)

        # Log normalization
        if name not in z_score_only:
            # Apply training offset
            n_clipped = int((x + eps < floor).sum())

            # Record any non zeros
            if n_clipped > 0 and split_name != 'train':
                clip_report[split_name][name] = n_clipped

            # Shift, clip, and take log
            x = np.log10(np.maximum(x + eps, floor))

        # Z score normalixation
        X[:, :, j] = (x - params['mean']) / params['std']

# Any NaNs get picked up here
for split_name, X in splits.items():
    assert np.isfinite(X).all(), f"{split_name} has NaN/Inf after normalization"

# Record any that required shifting
shifted_feats = [n for n, p in norm_params.items() if p.get('shifted')]
print("Features shifted to positive before log10:", shifted_feats or "None")


for split_name, rep in clip_report.items():
    if rep:
        total = splits[split_name].shape[0] * splits[split_name].shape[1]
        print(f"\n{total} {split_name} values clipped at log floor (below training range):")
        for n, c in rep.items():
            print(f"  {n}: {c} / {total} ({100.0 * c / total:.4f}%)")

print()
print("==========================")
print("HYBRID NORMALIZATION STATS")
print("==========================")
for name, params in norm_params.items():
    print(f"\n{name}:")
    for k, v in params.items():
        formatted_val = round(v,5) if isinstance(v, (float, np.floating)) else v
        print(f"  {k:<8}: {formatted_val}")

Features shifted to positive before log10: None

512640 validation values clipped at log floor (below training range):
  Xl: 1740 / 512640 (0.3394%)
  Xs: 1111 / 512640 (0.2167%)

1024992 test values clipped at log floor (below training range):
  Xl: 1206 / 1024992 (0.1177%)
  Xs: 16 / 1024992 (0.0016%)

HYBRID NORMALIZATION STATS

F:
  method  : log_zscore
  eps     : 1.49
  floor   : 1.76
  shifted : False
  mean    : 1.04812
  std     : 0.64968

Np:
  method  : log_zscore
  eps     : 0.23
  floor   : 0.26
  shifted : False
  mean    : 0.68984
  std     : 0.29806

P4:
  method  : log_zscore
  eps     : 0.00281
  floor   : 0.00281
  shifted : False
  mean    : -1.58292
  std     : 0.54222

P5:
  method  : log_zscore
  eps     : 0.00336
  floor   : 0.00336
  shifted : False
  mean    : -1.82731
  std     : 0.30069

P6:
  method  : log_zscore
  eps     : 0.00138
  floor   : 0.00196
  shifted : False
  mean    : -2.3274
  std     : 0.16304

Tp:
  method  : log_zscore
  eps     : 6005.0
 

In [8]:
# Save normalized data and params to file
output_dir = Path('../processed')
output_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    output_dir / 'normalized_split.npz',
    X_train=X_train_norm,
    X_val=X_val_norm,
    X_test=X_test_norm,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test
)

with open(output_dir / 'norm_params.pkl', 'wb') as f:
    pickle.dump(norm_params, f)

print()
print(f"Files successfully saved to: {output_dir.resolve()}")


Files successfully saved to: C:\Users\brock\Documents\pCloud Documents\Schoolwork\HiDAM\SEP\processed
